# Task 1: Data Pipeline Construction (with mandatory logging)

In [1]:
import logging

log_file = r"C:\Users\symev\OneDrive\Training\Emeritus\capstone project\emeritus captone project\part2_python\pipeline.log"

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

logger.handlers.clear()

file_handler = logging.FileHandler(
    log_file,
    mode="w",
    encoding="utf-8"
)

formatter = logging.Formatter(
    "{asctime} - {levelname} - {name} - {message}",
    style="{",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Test that logging is working
logger.info("Logging started successfully.")

In [2]:
import os 
os.getcwd()

'C:\\Users\\symev\\OneDrive\\Training\\Emeritus\\capstone project\\emeritus captone project\\part2_python'

In [3]:
import pandas as pd
import numpy as np

In [4]:
file_path = r"C:\Users\symev\OneDrive\Training\Emeritus\capstone project\emeritus captone project\Metro_Interstate_Traffic_Volume.csv"

try:
    # Load the raw CSV file
    df = pd.read_csv(
        file_path,
        header=0
    )

    # Get number of rows and columns
    rows, columns = df.shape

    # Log successful data loading
    logger.info(
        "Raw CSV file loaded successfully: %d rows and %d columns.",
        rows,
        columns
    )

    # Display confirmation
    print("\nFile loaded successfully!")
    print(f"Rows: {rows}")
    print(f"Columns: {columns}")
    print("\nFirst 5 rows:")
    print(df.head())

except FileNotFoundError:
    logger.error(
        "File not found: '%s'.",
        file_path
    )

except pd.errors.EmptyDataError:
    logger.error(
        "The CSV file is empty."
    )

except pd.errors.ParserError:
    logger.error(
        "The CSV file could not be parsed. Please check its format."
    )

except PermissionError:
    logger.error(
        "Permission denied when trying to access '%s'.",
        file_path
    )

except OSError as e:
    logger.error(
        "Error reading the file: %s",
        e
    )


File loaded successfully!
Rows: 48204
Columns: 9

First 5 rows:
  holiday    temp  rain_1h  snow_1h  clouds_all weather_main  \
0    None  288.28      0.0      0.0          40       Clouds   
1    None  289.36      0.0      0.0          75       Clouds   
2    None  289.58      0.0      0.0          90       Clouds   
3    None  290.13      0.0      0.0          90       Clouds   
4    None  291.14      0.0      0.0          75       Clouds   

  weather_description            date_time  traffic_volume  
0    scattered clouds  2012-10-02 09:00:00            5545  
1       broken clouds  2012-10-02 10:00:00            4516  
2     overcast clouds  2012-10-02 11:00:00            4767  
3     overcast clouds  2012-10-02 12:00:00            5026  
4       broken clouds  2012-10-02 13:00:00            4918  


In [5]:
#checking df schema and expected columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   holiday              48204 non-null  object 
 1   temp                 48204 non-null  float64
 2   rain_1h              48204 non-null  float64
 3   snow_1h              48204 non-null  float64
 4   clouds_all           48204 non-null  int64  
 5   weather_main         48204 non-null  object 
 6   weather_description  48204 non-null  object 
 7   date_time            48204 non-null  object 
 8   traffic_volume       48204 non-null  int64  
dtypes: float64(3), int64(2), object(4)
memory usage: 3.3+ MB


In [6]:
# check df summary statistics
df.describe()

,temp,rain_1h,snow_1h,clouds_all,traffic_volume
count,48204.000000,48204.000000,48204.000000,48204.000000,48204.000000
mean,281.205870,0.334264,0.000222,49.362231,3259.818355
std,13.338232,44.789133,0.008168,39.015750,1986.860670
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,272.160000,0.000000,0.000000,1.000000,1193.000000
50%,282.450000,0.000000,0.000000,64.000000,3380.000000
75%,291.806000,0.000000,0.000000,90.000000,4933.000000
max,310.070000,9831.300000,0.510000,100.000000,7280.000000


In [7]:
# Categorical columns to standardise
categorical_columns = [
    'holiday',
    'weather_main',
    'weather_description'
]

# Standardise categorical values
for column in categorical_columns:

    # Keep a copy of the original values
    original_values = df[column].copy()

    # Remove leading/trailing spaces and standardise capitalisation
    df[column] = (
        df[column]
        .astype('string')
        .str.strip()
        .str.title()
    )

    # Count how many rows were modified
    rows_modified = (
        original_values.astype('string') != df[column]
    ).sum()

    # Log changes
    if rows_modified > 0:
        logger.warning(
            "Modified %d rows in '%s'. Reason: standardised "
            "categorical values by removing extra spaces and "
            "standardising capitalisation.",
            rows_modified,
            column
        )

print("Categorical data standardisation completed.")

Categorical data standardisation completed.


In [8]:
# Parse and validate date/time field

# Convert date_time to datetime
df['date_time'] = pd.to_datetime(
    df['date_time'],
    errors='coerce'
)

# Count values that could not be parsed
invalid_datetime = df['date_time'].isna().sum()

# Log invalid date/time values
if invalid_datetime > 0:
    logger.warning(
        "Found %d invalid date/time values. "
        "Reason: values could not be parsed as valid dates/times.",
        invalid_datetime
    )

    # Remove rows with invalid date/time values
    df = df.dropna(subset=['date_time'])

    logger.warning(
        "Dropped %d rows. "
        "Reason: date/time values could not be parsed as valid dates/times.",
        invalid_datetime
    )
else:
    logger.info(
        "All date/time values parsed successfully."
    )

# Display the result
print("\nDate/time validation:")
print(f"Valid date/time values: {df['date_time'].notna().sum()}")
print(f"Invalid date/time values removed: {invalid_datetime}")
print(f"Date/time data type: {df['date_time'].dtype}")


Date/time validation:
Valid date/time values: 48204
Invalid date/time values removed: 0
Date/time data type: datetime64[ns]


In [9]:
#checking df schema to ensure datetime type is correct 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              48204 non-null  string        
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  string        
 6   weather_description  48204 non-null  string        
 7   date_time            48204 non-null  datetime64[ns]
 8   traffic_volume       48204 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(2), string(3)
memory usage: 3.3 MB


In [10]:
# Identify and remove duplicate rows

# Count duplicate rows before removing them
duplicate_rows = df.duplicated().sum()

if duplicate_rows > 0:

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Log the number of rows removed and the reason
    logger.warning(
        "Dropped %d duplicate rows. "
        "Reason: duplicate records were identified.",
        duplicate_rows
    )

    print(f"\nRemoved {duplicate_rows} duplicate rows.")

else:

    # Log that no duplicates were found
    logger.info(
        "No duplicate rows were found."
    )

    print("\nNo duplicate rows found.")

# Confirm the final number of rows
print(f"Rows remaining: {len(df)}")



Removed 17 duplicate rows.
Rows remaining: 48187


In [11]:
# Check for impossible temperature values

invalid_temp = df['temp'] == 0
temp_rows = invalid_temp.sum()

if temp_rows > 0:

    # Remove rows with invalid temperature values
    df = df[~invalid_temp]

    # Log the number of rows removed and the reason
    logger.warning(
        "Dropped %d rows. Reason: temperature value of 0 "
        "was identified as an invalid/impossible reading.",
        temp_rows
    )

    print(
        f"Dropped {temp_rows} rows with invalid temperature values."
    )

else:

    # Log that no invalid values were found
    logger.info(
        "No invalid temperature values were found."
    )

Dropped 10 rows with invalid temperature values.


In [12]:
#Check for impossible rainfall values

invalid_rain = df['rain_1h'] > 9000
rain_rows = invalid_rain.sum()

if rain_rows > 0:
    df = df[~invalid_rain]

    logging.warning(
        "Dropped %d rows. Reason: rainfall value above 9000 "
        "was identified as an invalid/impossible reading.",
        rain_rows
    )

    print(f"Dropped {rain_rows} rows with invalid rainfall values.")

else:
    logging.info("No invalid rainfall values were found.")


Dropped 1 rows with invalid rainfall values.


In [13]:
# check df summary statistics to ensure that outliers have been removed
df.describe()

,temp,rain_1h,snow_1h,clouds_all,traffic_volume
count,48176.000000,48176.000000,48176.000000,48176.000000,48176.000000
mean,281.262931,0.130388,0.000223,49.375166,3259.973887
std,12.709672,1.003654,0.008170,39.013008,1986.847829
min,243.390000,0.000000,0.000000,0.000000,0.000000
25%,272.180000,0.000000,0.000000,1.000000,1194.000000
50%,282.460000,0.000000,0.000000,64.000000,3380.000000
75%,291.807000,0.000000,0.000000,90.000000,4933.000000
max,310.070000,55.630000,0.510000,100.000000,7280.000000


In [14]:
logging.info(
    "Data cleaning completed successfully. "
    "Categorical values were standardised, date/time fields validated, "
    "duplicates removed, and invalid values handled."
)

print("Data cleaning completed successfully.")

Data cleaning completed successfully.


In [15]:
logger.info(
    "Data cleaning completed successfully. "
    "Categorical values were standardised, date/time fields validated, "
    "duplicates removed, and invalid values handled."
)

print("Data cleaning completed successfully.")

INFO:__main__:Data cleaning completed successfully. Categorical values were standardised, date/time fields validated, duplicates removed, and invalid values handled.


Data cleaning completed successfully.
